In [4]:
import numpy as np
import mocaxpy
from inception.quantlib_python import VanillaOption
import time

In [7]:
def option_pricer(stock_price: float,
                  strike_price: float, 
                  vol: float,
                  time_to_maturity: int,
                  free_rate: float,
                  option_style: str = 'European',
                  direction: str = 'Put'):
    """
    The Option pricer function wrapper, (adapts the original function signature to the signature expected by MoCaX)
    :param x: [stock price, strike, maturity, volatility]
    """
    init_date = '2022-01-01'
    
    option = VanillaOption(direction,
                           init_date,
                           exercise_date=time_to_maturity,
                           stock_price=stock_price,
                           strike=strike_price,
                           vol=vol,
                           free_rate=free_rate,
                           style=option_style)
    
    return option.value


In [9]:
option_pricer(10, 10, 0.2, 1, 0.05)

0.04107882635153256

In [21]:
@timer
def option_chebyshev_approximate(time_to_maturity, pricer: 'function'):
    """
    Using Chebyshev Tensor to approximate the Autocallable notes pricer
    :param evaluate_date: 
    :return:
    """
    import pandas as pd
    import mocaxpy
    import numpy as np
    import os
    # Number of dimensions
    num_dimensions = 3
    free_rate = 0.05
    direction = 'Call'
    style = 'European'

    # Function domain for stock A, B and C price, Time to Maturity
    issued_price = [382.82, 494.08, 142.86]

    lower_bound = 0.2
    upper_bound = 1.1

    domain_values = [
        [200, 750],  # Stock price
        [200, 750],  # Strike
        [0.1, 0.5]   # Volatility
    ]
    domain = mocaxpy.MocaxDomain(domain_values)

    # MoCaX accuracy parameters, Chebyshev Nodes
    n_nodes = [30, 30, 7]
    mocax_nodes = mocaxpy.MocaxNs(n_nodes)

    # Maximum derivative order.
    max_derivative_order = 2

    option_mocax = mocaxpy.Mocax(None, 
                                 num_dimensions,
                                 domain, 
                                 None, 
                                 mocax_nodes,
                                 max_derivative_order=max_derivative_order)
    # Get the Chebysheve points
    chebysheve_points = option_mocax.get_evaluation_points()
    
    y = [pricer(x[0], x[1], x[2], time_to_maturity, free_rate, style, direction) for x in chebysheve_points]
    # Set the y to Chebysheve object
    option_mocax.set_original_function_values(y)
    
    file_dir = f'./OptionB_chebyshev_database/'
    if not os.path.exists(file_dir):
        os.makedirs(file_dir)
        
    file_name = file_dir + f'{direction}_{time_to_maturity}.mcx'
    option_mocax.serialize(file_name)
    
    
def create_chebyshev_approximator(start_date: int, end_date: int, chebyshev_approximator: 'function', pricer: 'function'):
    """
    Create chebyshev object daily
    """
    import pandas as pd
    from tqdm import tqdm
    
    date_range = range(start_date, end_date)
    for d in tqdm(date_range):
        chebyshev_approximator(d, pricer)   

In [22]:
create_chebyshev_approximator(1, 61, option_chebyshev_approximate, option_pricer)

  2%|█▍                                                                                 | 1/60 [00:02<02:40,  2.71s/it]

***option_chebyshev_approximate running time: 2.714 s***


  3%|██▊                                                                                | 2/60 [00:05<02:40,  2.78s/it]

***option_chebyshev_approximate running time: 2.818 s***


  5%|████▏                                                                              | 3/60 [00:08<02:35,  2.72s/it]

...

***option_chebyshev_approximate running time: 2.766 s***


 98%|████████████████████████████████████████████████████████████████████████████████▋ | 59/60 [02:41<00:03,  3.04s/it]

***option_chebyshev_approximate running time: 3.782 s***


100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [02:45<00:00,  2.76s/it]

***option_chebyshev_approximate running time: 4.337 s***


In [23]:
@timer
def option_chebyshev_approximate(time_to_maturity, pricer: 'function'):
    """
    Using Chebyshev Tensor to approximate the Autocallable notes pricer
    :param evaluate_date: 
    :return:
    """
    import pandas as pd
    import mocaxpy
    import numpy as np
    import os
    # Number of dimensions
    num_dimensions = 3
    free_rate = 0.05
    direction = 'Call'
    style = 'European'

    # Function domain for stock A, B and C price, Time to Maturity
    issued_price = [382.82, 494.08, 142.86]

    lower_bound = 0.2
    upper_bound = 1.1

    domain_values = [
        [30, 250],  # Stock price
        [30, 250],  # Strike
        [0.1, 0.5]   # Volatility
    ]
    domain = mocaxpy.MocaxDomain(domain_values)

    # MoCaX accuracy parameters, Chebyshev Nodes
    n_nodes = [30, 30, 7]
    mocax_nodes = mocaxpy.MocaxNs(n_nodes)

    # Maximum derivative order.
    max_derivative_order = 2

    option_mocax = mocaxpy.Mocax(None, 
                                 num_dimensions,
                                 domain, 
                                 None, 
                                 mocax_nodes,
                                 max_derivative_order=max_derivative_order)
    # Get the Chebysheve points
    chebysheve_points = option_mocax.get_evaluation_points()
    
    y = [pricer(x[0], x[1], x[2], time_to_maturity, free_rate, style, direction) for x in chebysheve_points]
    # Set the y to Chebysheve object
    option_mocax.set_original_function_values(y)
    
    file_dir = f'./OptionC_chebyshev_database/'
    if not os.path.exists(file_dir):
        os.makedirs(file_dir)
        
    file_name = file_dir + f'{direction}_{time_to_maturity}.mcx'
    option_mocax.serialize(file_name)
    
    
def create_chebyshev_approximator(start_date: int, end_date: int, chebyshev_approximator: 'function', pricer: 'function'):
    """
    Create chebyshev object daily
    """
    import pandas as pd
    from tqdm import tqdm
    
    date_range = range(start_date, end_date)
    for d in tqdm(date_range):
        chebyshev_approximator(d, pricer)   

In [24]:
create_chebyshev_approximator(1, 61, option_chebyshev_approximate, option_pricer)

  2%|█▍                                                                                 | 1/60 [00:02<02:51,  2.92s/it]

***option_chebyshev_approximate running time: 2.915 s***


  3%|██▊                                                                                | 2/60 [00:05<02:51,  2.95s/it]

***option_chebyshev_approximate running time: 2.983 s***


  5%|████▏                                                                              | 3/60 [00:08<02:44,  2.88s/it]

...

***option_chebyshev_approximate running time: 2.934 s***


 98%|████████████████████████████████████████████████████████████████████████████████▋ | 59/60 [02:56<00:03,  3.11s/it]

***option_chebyshev_approximate running time: 2.690 s***


100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [02:59<00:00,  2.99s/it]

***option_chebyshev_approximate running time: 2.662 s***


In [ ]:
## Anil

# Please run three times, and remove the comments in different intervals each time
# --------------------------------------------------
time_bucket_1 = (0, autocallable_note_pricer)
time_bucket_2 = (1, autocallable_note_pricer)
time_bucket_3 = (2, autocallable_note_pricer)
time_bucket_4 = (3, autocallable_note_pricer)

# time_bucket_1 = (6, autocallable_note_pricer)
# time_bucket_2 = (7, autocallable_note_pricer)
# time_bucket_3 = (8, autocallable_note_pricer)
# time_bucket_4 = (9, autocallable_note_pricer)

# time_bucket_1 = (10, autocallable_note_pricer)
# time_bucket_2 = (11, autocallable_note_pricer)
# time_bucket_3 = (12, autocallable_note_pricer)
# time_bucket_4 = (13, autocallable_note_pricer)

# time_bucket_1 = (14, autocallable_note_pricer)
# time_bucket_2 = (15, autocallable_note_pricer)

# --------------------------------------------------

p1 = multiprocess.Process(target=autocallable_notes_chebyshev_approximate, args=time_bucket_1)
p2 = multiprocess.Process(target=autocallable_notes_chebyshev_approximate, args=time_bucket_2)
p3 = multiprocess.Process(target=autocallable_notes_chebyshev_approximate, args=time_bucket_3)
p4 = multiprocess.Process(target=autocallable_notes_chebyshev_approximate, args=time_bucket_4)

# starting process 1
p1.start()
# starting process 2
p2.start()
# starting process 3
p3.start()
# starting process 3
p4.start()

# wait until process 1 is finished
p1.join()
# wait until process 2 is finished
p2.join()
p3.join()
p4.join()

# both processes finished
print("Done!")